# Cultivar-conditional probability blending

The notebook records the experiment configuration, analysis procedure and saved outputs for this study.


## 1. Validate original manifests, archived C11 decisions and prior experiment completion


In [ ]:
from __future__ import annotations
import os, json, hashlib, time, math, platform
from pathlib import Path
import numpy as np
import pandas as pd
import cv2
import torch
from torch import nn
from torchvision import models
from scipy.special import softmax
from sklearn.linear_model import LogisticRegression
from sklearn.preprocessing import StandardScaler
from sklearn.pipeline import Pipeline
from sklearn.metrics import accuracy_score, f1_score, log_loss, confusion_matrix
from sklearn.utils.validation import check_is_fitted
from tqdm.auto import tqdm
from IPython.display import display
import matplotlib.pyplot as plt
from PIL import Image, ImageOps

REVISION='V51-DANDHI-TARGETED-RGB-R1-20261023'
V4=Path('external_data')
STAGE1=Path('external_data')
SPATIAL=STAGE1/'v51_locked_spatial'
PILOT=STAGE1/'v51_grader_decision_pilot'
PREVIOUS=STAGE1/'v51_representation_finetune_pilot'
OUT=STAGE1/'v51_dandhi_targeted_rgb_pilot'
SEEDS=(11,23,37)
LABELS=('Small','Medium','Large')
LABEL_ID={label:i for i,label in enumerate(LABELS)}
PRIMARY=0.10
BOOTSTRAP_REPS=1600
BLEND_WEIGHTS=(0.0,0.15,0.30,0.50)   # fixed, development-only correction
MIN_DEV_MACRO_F1_GAIN=0.005
MIN_DANDHI_DEV_MACRO_F1_GAIN=0.01
SPECIALIST_SEED=11
ORDINAL_TRAIN_SEED=20261023
ORDINAL_EPOCHS=160
ORDINAL_PATIENCE=18
ORDINAL_WEIGHT_DECAY=0.01
ORDINAL_LR=0.002
BATCH=48
DEVICE=torch.device('cpu')  # CPU execution for this analysis stage
torch.set_num_threads(min(8,max(1,os.cpu_count() or 1)))

def must(condition,message):
    if not bool(condition):raise RuntimeError(message)
def text_csv(path):
    path=Path(path); must(path.is_file(),f'Missing required input: {path}')
    return pd.read_csv(path,dtype=str,keep_default_na=False,low_memory=False)
def sha(path):
    h=hashlib.sha256()
    with open(path,'rb') as f:
        for block in iter(lambda:f.read(1048576),b''):h.update(block)
    return h.hexdigest()
def row_digest(frame,columns):
    h=hashlib.sha256()
    for values in frame[list(columns)].itertuples(index=False,name=None):
        h.update(json.dumps([str(v) for v in values],separators=(',',':')).encode()+b'\n')
    return h.hexdigest()
def json_write(path,payload):
    path=Path(path)
    p=path.with_name(path.name+'.partial')
    p.write_text(json.dumps(payload,indent=2,sort_keys=True,default=str),encoding='utf-8')
    os.replace(p,path)

must(all(p.is_dir() for p in (V4,STAGE1,SPATIAL,PILOT,PREVIOUS)),'Required experiment folders missing')
must((SPATIAL/'RESULTS_COMPLETE.json').is_file(),'V5.1 spatial experiment incomplete')
must((PILOT/'RESULTS_COMPLETE.json').is_file(),'Improved-grader experiment incomplete')
must((PREVIOUS/'RESULTS_COMPLETE.json').is_file(),'Prior RGB representation experiment incomplete')
must((STAGE1/'verified_v4_image_path_map.csv').is_file(),'SHA-verified V4 photograph map missing')
must(OUT not in (V4,STAGE1,SPATIAL,PILOT,PREVIOUS) and V4 not in OUT.parents and SPATIAL not in OUT.parents and PREVIOUS not in OUT.parents,
     'Output path risks overwriting frozen evidence')
OUT.mkdir(exist_ok=True,parents=True)
source=text_csv(V4/'data/source_split.csv')
source=source[source.eligibility.eq('eligible')].copy()
must(len(source)==5828 and source.source_id.is_unique,'V4 source manifest changed')
must(source.split.value_counts().to_dict()=={'train':3603,'development':835,'calibration':556,'locked_test':834},
     'Frozen source partitions differ')
must(source.groupby('source_group').split.nunique().max()==1,'Source-group split leakage')
map_df=text_csv(STAGE1/'verified_v4_image_path_map.csv')
must(map_df.source_id.is_unique and len(map_df)==5828,'SHA mapping count/IDs invalid')
src=source.merge(map_df[['source_id','file_sha256','relative_path','verified_image_path']],
                 on='source_id',how='left',validate='one_to_one',suffixes=('','_map'),indicator=True)
must(src._merge.eq('both').all(),'Incomplete image mapping')
for key in ('file_sha256','relative_path'):
    must(src[key].eq(src[f'{key}_map']).all(),f'Hash mapping mismatch: {key}')
src=src.drop(columns=['_merge','file_sha256_map','relative_path_map']).reset_index(drop=True)
must(src.verified_image_path.map(lambda x:Path(x).is_file()).all(),'A verified source photograph no longer exists')

def score_frame(which):
    is_cal=which=='calibration'
    csv=V4/('calibration/scores/C11_1049.csv' if is_cal else 'results/C11_1049.csv')
    folder=V4/('calibration/benchmark' if is_cal else 'results/fixed_benchmark')
    frame=text_csv(csv)
    b=text_csv(folder/'manifest.csv')
    n=1668 if is_cal else 12510
    must(len(frame)==len(b)==n and frame.sample_id.is_unique and b.sample_id.is_unique,
         f'{which}: benchmark row count or uniqueness mismatch')
    must(set(frame.sample_id)==set(b.sample_id),f'{which}: score/benchmark samples differ')
    aligned=b.set_index('sample_id').loc[frame.sample_id]
    for key in ('source_id','source_group','source_size','source_variety','kind','label','image_sha256'):
        must(np.array_equal(frame[key].to_numpy(),aligned[key].to_numpy()),f'{which}: {key} mismatch')
    lookup=src[src.split.eq('calibration' if is_cal else 'locked_test')].set_index('source_id')
    must(set(frame.source_id)==set(lookup.index),f'{which}: wrong source set')
    must(frame.source_size.eq(frame.source_id.map(lookup.native_size_label)).all(),f'{which}: size label mismatch')
    must(frame.source_group.eq(frame.source_id.map(lookup.source_group)).all(),f'{which}: source group mismatch')
    frame['verified_path']=[str(folder/'images'/f'{sample_id}.png') for sample_id in frame.sample_id]
    must(frame.verified_path.map(lambda p:Path(p).is_file()).all(),f'{which}: benchmark image missing')
    must(frame.groupby('source_id').size().eq(3 if is_cal else 15).all(),f'{which}: incomplete variant families')
    return frame
cal=score_frame('calibration');test=score_frame('locked_test')
must(int(test.label.eq('1').sum())==10008 and int(test.kind.eq('original').sum())==834,
     'Test synthetic/original record counts differ')
must(int(test.kind.str.startswith('benign').sum())==1668,'Benign record count differs')
must(cal.label.eq('0').all() and cal.kind.eq('original').sum()==556,'Calibration status differs')
existing=text_csv(PILOT/'v51_improved_grader_all_variant_predictions.csv')
must(existing.sample_id.is_unique and len(existing)==12510,'Prior ensemble predictions missing/incomplete')
existing=existing.set_index('sample_id').loc[test.sample_id].reset_index()
for key in ('source_id','source_group','kind','label','source_size'):
    must(existing[key].eq(test[key]).all(),f'Prior ensemble {key} diverges from frozen benchmark')

print('PASS: 5,828 verified-source metadata rows; 1,668 calibration and 12,510 locked-test variants.')
print('PASS: previous size-ensemble predictions and V4 C11 identities align.')
# EARLY frozen-C11 replay check, before expensive geometry extraction or grading.
# The completed baseline calibrated on empirical ranks, not directly on raw C11 scores.
def _rank_replay_c11(budget):
    cal_original=cal.kind.eq('original').to_numpy()
    normal=np.sort(pd.to_numeric(cal.score,errors='raise').to_numpy(np.float64)[cal_original])
    must(len(normal)==556 and np.isfinite(normal).all(), 'Invalid original C11 calibration reference')
    ref_ranks=np.searchsorted(normal,normal,side='right')/len(normal)
    cutoff=np.quantile(ref_ranks,1-budget,method='higher')
    test_ranks=np.searchsorted(normal,pd.to_numeric(test.score,errors='raise').to_numpy(np.float64),side='right')/len(normal)
    return test_ranks>cutoff
for _budget in (.05,.10,.15):
    _column=f'refer_C11_{int(round(100*_budget))}'
    must(_column in existing.columns,f'Early preflight: archived {_column} missing')
    _arch=existing[_column].astype(str).str.strip().str.lower()
    must(_arch.isin(['true','false','1','0']).all(),f'Early preflight: malformed {_column}')
    _saved=_arch.isin(['true','1']).to_numpy(dtype=bool)
    _replayed=_rank_replay_c11(_budget)
    _difference=int((_saved!=_replayed).sum())
    print(f'EARLY C11 {int(_budget*100)}% archived/rank-replayed decision mismatch: {_difference} / {len(test)}')
    # Verify risk-score ordering even if archival/recalculated decisions disagree
    # near the calibration boundary due to changed numerical precision.
    _anom=pd.to_numeric(test.score,errors='raise').to_numpy(np.float64)
    must(_saved.any() and (~_saved).any() and _anom[~_saved].max()<=_anom[_saved].min()+1e-10,
         f'Archived {_column} decisions violate the frozen anomaly-score ordering')

print('Execution device:',DEVICE,'| Output:',OUT)
print('Variety-specific routing requires externally available variety metadata at inference time.')


## 2. Restore the frozen incumbent and establish development controls


In [ ]:
from torch.utils.data import Dataset, DataLoader
from torchvision import transforms
from urllib.parse import urlparse
import copy, random

from scipy.special import softmax
from sklearn.metrics import f1_score, accuracy_score, log_loss, confusion_matrix

TRAIN=src.loc[src.split.eq('train')].reset_index(drop=True)
DEV=src.loc[src.split.eq('development')].reset_index(drop=True)
LABEL_ID={label:i for i,label in enumerate(LABELS)}
y_train=np.array([LABEL_ID[x] for x in TRAIN.native_size_label],dtype=np.int64)
y_dev=np.array([LABEL_ID[x] for x in DEV.native_size_label],dtype=np.int64)
var_dev=DEV.variety.to_numpy(dtype=str)
dandhi_dev=var_dev=='Dandhi'
must(int(dandhi_dev.sum())>100,'Expected Dandhi development observations')

choice=json.loads((PILOT/'development_model_selection.json').read_text())
selected_name=str(choice['winner']); T=float(choice['temperature'])
must(.1<=T<=20,'Previous temperature invalid')

def row_fingerprint(rows,which):
    if which in ('train','development'):
        keys=rows[['source_id','file_sha256']].rename(columns={'source_id':'id','file_sha256':'sha'})
    else:
        keys=rows[['sample_id','image_sha256']].rename(columns={'sample_id':'id','image_sha256':'sha'})
    return row_digest(keys,['id','sha'])

def layer3_cache(part,rows):
    fp=SPATIAL/f'fixed_layer3_{part}.npy'
    meta=SPATIAL/f'fixed_layer3_{part}.json'
    must(fp.is_file() and meta.is_file(),f'Missing frozen layer3 cache for {part}')
    info=json.loads(meta.read_text())
    must(info['row_sha256']==row_fingerprint(rows,part),f'{part}: SHA/order cache mismatch')
    x=np.load(fp,mmap_mode='r',allow_pickle=False)
    must(x.shape==(len(rows),256,14,14) and x.dtype==np.float16,f'{part}: unexpected cache shape')
    return x

class SizeHead(nn.Module):
    def __init__(self):
        super().__init__()
        self.layer4=models.resnet18(weights=None).layer4
        self.fc=nn.Linear(512,3)
    def forward(self,x):
        return self.fc(self.layer4(x).mean((-2,-1)))

@torch.inference_mode()
def head_logits(net,features,batch=96):
    net.eval()
    output=[]
    for i in range(0,len(features),batch):
        xx=torch.from_numpy(np.asarray(features[i:i+batch],dtype=np.float32).copy())
        output.append(net(xx).detach().cpu().numpy())
    return np.concatenate(output,axis=0)

selected_models={}
for seed in SEEDS:
    ck=SPATIAL/f'size_layer4_seed{seed}.pt'
    must(ck.is_file(),f'Missing saved seed model: {ck}')
    payload=torch.load(ck,map_location='cpu',weights_only=True)
    must(payload['config']['seed']==seed,'Seed metadata mismatch')
    m=SizeHead(); m.load_state_dict(payload['model_state'],strict=True);m.eval()
    selected_models[seed]=m

x_dev=layer3_cache('development',DEV)
p_dev_by_seed=np.stack([softmax(head_logits(selected_models[k],x_dev),axis=1) for k in SEEDS])
def selected_baseline(raw_p):
    if selected_name=='equal_weight_ensemble': p=raw_p.mean(axis=0)
    else:
        must(selected_name.startswith('seed_'),f'Unexpected previous winner: {selected_name}')
        seed=int(selected_name[5:]);must(seed in SEEDS,'Unknown selected seed')
        p=raw_p[SEEDS.index(seed)]
    return softmax(np.log(np.clip(p,1e-12,1))/T,axis=1)

p_base_dev=selected_baseline(p_dev_by_seed)
base_dev_f1=f1_score(y_dev,p_base_dev.argmax(axis=1),average='macro')
base_dandhi_f1=f1_score(y_dev[dandhi_dev],p_base_dev.argmax(axis=1)[dandhi_dev],average='macro')
print('PASS: archived ensemble DEV control:',selected_name,
      'macro F1',round(base_dev_f1,4),'Dandhi macro F1',round(base_dandhi_f1,4))

WEIGHTS=models.ResNet18_Weights.IMAGENET1K_V1
local_weight_path=Path(torch.hub.get_dir())/'checkpoints'/Path(urlparse(WEIGHTS.url).path).name
must(local_weight_path.is_file(),f'Missing verified local ImageNet weights: {local_weight_path}')
weight_hash=sha(local_weight_path)
existing_head=selected_models[11]
net=models.resnet18(weights=None)
net.load_state_dict(torch.load(local_weight_path,map_location='cpu',weights_only=True),strict=True)
net.layer4.load_state_dict(existing_head.layer4.state_dict(),strict=True)
# torchvision's ImageNet checkpoint has a 1,000-way classification head.
# Replace it with the three-class date-size head BEFORE loading the saved weights.
assert net.fc.in_features == existing_head.fc.in_features == 512
net.fc = nn.Linear(net.fc.in_features, len(LABELS))
net.fc.load_state_dict(existing_head.fc.state_dict(),strict=True)
assert net.fc.out_features == len(LABELS) == 3
assert all(torch.equal(net.fc.state_dict()[key], existing_head.fc.state_dict()[key])
           for key in ('weight', 'bias')), 'Restored grading head differs from saved checkpoint'
assert all(torch.equal(net.layer4.state_dict()[key], existing_head.layer4.state_dict()[key])
           for key in existing_head.layer4.state_dict()), 'Restored layer4 differs from saved checkpoint'
net.eval()
print('PASS: ImageNet ResNet-18 backbone and saved three-class grading head restored.')


## 3. Restore, do not retrain, the previously saved RGB fine-tuned model


In [ ]:
PREPROCESS=transforms.Compose([
    transforms.Resize((224,224),interpolation=transforms.InterpolationMode.BILINEAR),
    transforms.ToTensor(),
    transforms.Normalize((.485,.456,.406),(.229,.224,.225))
])

def canonical(path):
    with Image.open(path) as im:
        im=ImageOps.exif_transpose(im).convert('RGB')
        im.thumbnail((256,256),Image.Resampling.BICUBIC)
        canvas=Image.new('RGB',(256,256),(128,128,128))
        canvas.paste(im,((256-im.width)//2,(256-im.height)//2))
    return canvas

class ImageRows(Dataset):
    def __init__(self,paths,identifiers,canonical_input=False):
        self.paths=[str(x) for x in paths]
        self.ids=[str(x) for x in identifiers]
        self.canonical_input=bool(canonical_input)
        must(len(self.paths)==len(self.ids),'Dataset length mismatch')
    def __len__(self):return len(self.ids)
    def __getitem__(self,i):
        if self.canonical_input: image=canonical(self.paths[i])
        else:
            with Image.open(self.paths[i]) as im:image=im.convert('RGB')
        return PREPROCESS(image),self.ids[i]

@torch.inference_mode()
def logits_for_loader(model,ds,batch=48):
    loader=DataLoader(ds,batch_size=batch,shuffle=False,num_workers=0,pin_memory=False)
    model.eval()
    logits=[];seen=[]
    for images,identifiers in loader:
        logits.append(model(images).cpu().numpy().astype(np.float32))
        seen.extend(list(identifiers))
    must(seen==ds.ids,'Image inference ordering changed')
    return np.concatenate(logits,axis=0)

model_file=PREVIOUS/'rgb_representation_finetuned.pt'
must(model_file.is_file(),'Missing saved RGB fine-tuned checkpoint; complete previous RGB notebook first')
candidate_ckpt_sha=sha(model_file)
payload=torch.load(model_file,map_location='cpu',weights_only=True)
prior_cfg=payload.get('config',{})
must(prior_cfg.get('train_source_fingerprint')==row_fingerprint(TRAIN,'train'),
     'RGB checkpoint does not belong to this training population')
must(prior_cfg.get('dev_source_fingerprint')==row_fingerprint(DEV,'development'),
     'RGB checkpoint does not belong to this development population')
must(prior_cfg.get('size_seed11_checkpoint_sha256')==sha(SPATIAL/'size_layer4_seed11.pt'),
     'RGB checkpoint initialized from a different grading head')
must(prior_cfg.get('starting_weight_sha256')==weight_hash,
     'RGB checkpoint initialized from a different pretrained backbone')
net.load_state_dict(payload['model_state'],strict=True)
net.eval()
for p in net.parameters():p.requires_grad_(False)

# Verify the complete development photograph set, not a filename-only sample.
for row in tqdm(DEV.itertuples(index=False),total=len(DEV),desc='Verify development photographs'):
    must(sha(row.verified_image_path)==row.file_sha256,
         f'Development image identity mismatch: {row.source_id}')
dev_data=ImageRows(DEV.verified_image_path,DEV.source_id,canonical_input=True)
logits_new_dev=logits_for_loader(net,dev_data)
p_new_dev=softmax(logits_new_dev/T,axis=1)
must(p_new_dev.shape==(835,3) and np.isfinite(p_new_dev).all(),
     'Malformed saved-RGB development inference')
print('PASS: reused previously trained RGB checkpoint, SHA-256',candidate_ckpt_sha[:20])
print('PASS: development data and model identity verified; no training executed')


## 4. Development-only choice: use RGB features exclusively for Dandhi


In [ ]:
DANDHI_WEIGHTS=(0.0,0.25,0.50,0.75,1.0)
MIN_OVERALL_GAIN=0.002
MIN_DANDHI_GAIN=0.010

def targeted_blend(base,candidate,alpha,is_dandhi):
    base=np.asarray(base,dtype=np.float64)
    candidate=np.asarray(candidate,dtype=np.float64)
    mask=np.asarray(is_dandhi,dtype=bool)
    must(base.shape==candidate.shape and base.ndim==2 and base.shape[1]==3,
         'Probability dimensions mismatch')
    must(len(mask)==len(base) and 0<=alpha<=1,'Variety mask or blend invalid')
    result=base.copy()
    result[mask]=(1-alpha)*base[mask]+alpha*candidate[mask]
    must(np.array_equal(result[~mask],base[~mask]),
         'Non-Dandhi predictions were altered')
    must(np.isfinite(result).all() and np.allclose(result.sum(axis=1),1,atol=1e-5),
         'Invalid probability mixture')
    return result

selection_rows=[]
for a in DANDHI_WEIGHTS:
    pdv=targeted_blend(p_base_dev,p_new_dev,a,dandhi_dev)
    selection_rows.append({
        'weight':float(a),
        'macro_f1':float(f1_score(y_dev,pdv.argmax(1),average='macro')),
        'dandhi_f1':float(f1_score(y_dev[dandhi_dev],pdv.argmax(1)[dandhi_dev],average='macro')),
        'accuracy':float(accuracy_score(y_dev,pdv.argmax(1))),
        'nll':float(log_loss(y_dev,pdv,labels=[0,1,2]))
    })
selection_table=pd.DataFrame(selection_rows)
allowed=selection_table.loc[
    (selection_table.weight>0) &
    (selection_table.macro_f1>=base_dev_f1+MIN_OVERALL_GAIN) &
    (selection_table.dandhi_f1>=base_dandhi_f1+MIN_DANDHI_GAIN)
]
if len(allowed):
    alpha=float(allowed.sort_values(['macro_f1','dandhi_f1','nll','weight'],
                                     ascending=[False,False,True,True]).iloc[0].weight)
else: alpha=0.0
selection={
    'revision':REVISION,'alpha':alpha,'selected_checkpoint_sha256':candidate_ckpt_sha,
    'incumbent_development_macro_f1':float(base_dev_f1),
    'incumbent_dandhi_development_macro_f1':float(base_dandhi_f1),
    'dev_results':selection_rows,'predefined_weights':list(DANDHI_WEIGHTS),
    'required_overall_macro_f1_gain':MIN_OVERALL_GAIN,
    'required_dandhi_macro_f1_gain':MIN_DANDHI_GAIN,
    'selection_uses_test_or_anomaly_labels':False,
    'prior_exposure_to_development_results':True
}
json_write(OUT/'development_only_selection.json',selection)
print('DEVELOPMENT DECISION:', 'TARGETED_ACCEPTED' if alpha>0 else 'TARGETED_REJECTED',
      'Dandhi-only blend weight',alpha)
display(selection_table.round(5))


## 5. C11 appearance signal and Dandhi RGB inference


In [ ]:
CAL=cal
TEST=test
CAL_CLEAN=CAL.kind.eq('original').to_numpy()
TEST_CLEAN=TEST.kind.eq('original').to_numpy()
TEST_BENIGN=TEST.kind.str.startswith('benign').to_numpy()
TEST_EDIT=TEST.label.eq('1').to_numpy()

def saved_flag(col):
    must(col in existing.columns,f'Archived policy column absent: {col}')
    data=existing[col].astype(str).str.strip().str.lower()
    must(data.isin(('true','false','0','1')).all(),f'Invalid archived boolean column: {col}')
    return data.isin(('true','1')).to_numpy()

old_conf_5=saved_flag('refer_confidence_5')
old_conf_10=saved_flag('refer_confidence_10')
c11_5=saved_flag('refer_C11_5')
c11_10=saved_flag('refer_C11_10')

def empirical_rank(cal_clean_scores,all_scores):
    xx=np.sort(np.asarray(cal_clean_scores,dtype=np.float64))
    yy=np.asarray(all_scores,dtype=np.float64)
    must(len(xx)==556 and np.isfinite(xx).all() and np.isfinite(yy).all(),
         'Invalid empirical rank reference')
    return np.searchsorted(xx,yy,side='right')/len(xx)

def calibrated_referral(cal_risk,test_risk,budget):
    ref=np.asarray(cal_risk,dtype=float)[CAL_CLEAN]
    cal_r=empirical_rank(ref,cal_risk)
    test_r=empirical_rank(ref,test_risk)
    threshold=np.quantile(cal_r[CAL_CLEAN],1-budget,method='higher')
    return test_r>threshold

@torch.inference_mode()
def predict_dandhi_cached(part,frame,chunk=128):
    is_dandhi=frame.source_variety.eq('Dandhi').to_numpy()
    records=frame.loc[is_dandhi].reset_index(drop=True)
    must(len(records)>0,'No Dandhi rows for '+part)
    prefix=OUT/'rgb_prediction_cache'/part
    prefix.mkdir(exist_ok=True,parents=True)
    outputs=[]
    for start in tqdm(range(0,len(records),chunk),desc='Dandhi RGB '+part):
        end=min(start+chunk,len(records))
        seg=records.iloc[start:end]
        ids=seg.sample_id.to_numpy(dtype=str)
        hashes=seg.image_sha256.to_numpy(dtype=str)
        paths=seg.verified_path.tolist()
        target=prefix/f'block_{start:06d}_{end:06d}.npz'
        for path,expected in zip(paths,hashes):
            must(sha(path)==expected,f'Edited/benign image SHA mismatch: {path}')
        if target.is_file():
            with np.load(target,allow_pickle=False) as z:
                must(str(z['ckpt_sha'])==candidate_ckpt_sha and
                     np.array_equal(z['ids'].astype(str),ids) and
                     np.array_equal(z['sha'].astype(str),hashes),
                     f'Stale RGB prediction cache: {target}')
                block=z['logits'].astype(np.float32)
        else:
            block=logits_for_loader(net,ImageRows(paths,ids,canonical_input=False),batch=32)
            must(block.shape==(end-start,3) and np.isfinite(block).all(),
                 f'Invalid RGB prediction chunk {part} {start}:{end}')
            temp=target.with_name(target.name+'.partial.npz')
            np.savez_compressed(temp,ids=ids,sha=hashes,ckpt_sha=np.asarray(candidate_ckpt_sha),
                                logits=block.astype(np.float32))
            os.replace(temp,target)
        must(block.shape==(end-start,3) and np.isfinite(block).all(),
             f'Invalid cached logits shape for {target}')
        outputs.append(block)
    result=np.concatenate(outputs,axis=0)
    must(result.shape==(int(is_dandhi.sum()),3),'Dandhi inference count mismatch')
    return result,is_dandhi

if alpha>0:
    # Evaluate only after the development-only accept decision.
    rgb_cal,mask_cal=predict_dandhi_cached('calibration',CAL)
    rgb_test,mask_test=predict_dandhi_cached('locked_test',TEST)
    must(mask_cal.sum()>0 and mask_test.sum()>0,'Missing target variety')

    def baseline_probs_cached(part,frame):
        features=layer3_cache(part,frame)
        raw=np.stack([softmax(head_logits(selected_models[k],features),axis=1) for k in SEEDS])
        return selected_baseline(raw)
    p_base_cal=baseline_probs_cached('calibration',CAL)
    p_base_test=baseline_probs_cached('locked_test',TEST)
    must(np.array_equal(np.asarray(LABELS)[p_base_test.argmax(1)],
                        existing.predicted_size.to_numpy()),
         'Restored archived grader class labels disagree')
    must(np.max(np.abs(p_base_test.max(1)-pd.to_numeric(existing.max_probability)))<.003,
         'Restored archived grader confidence disagrees')
    p_candidate_cal=p_base_cal.copy()
    p_candidate_test=p_base_test.copy()
    p_candidate_cal[mask_cal]=(1-alpha)*p_base_cal[mask_cal]+alpha*softmax(rgb_cal/T,axis=1)
    p_candidate_test[mask_test]=(1-alpha)*p_base_test[mask_test]+alpha*softmax(rgb_test/T,axis=1)
    must(np.array_equal(p_candidate_test[~mask_test],p_base_test[~mask_test]),
         'Unexpected change to non-Dandhi test probabilities')
    must(np.array_equal(p_candidate_cal[~mask_cal],p_base_cal[~mask_cal]),
         'Unexpected change to non-Dandhi calibration probabilities')
    for p in (p_candidate_cal,p_candidate_test):
        must(np.isfinite(p).all() and np.allclose(p.sum(1),1,atol=1e-5),
             'Invalid candidate probabilities')
    new_conf_5=calibrated_referral(1-p_candidate_cal.max(1),
                                   1-p_candidate_test.max(1),.05)
    new_conf_10=calibrated_referral(1-p_candidate_cal.max(1),
                                    1-p_candidate_test.max(1),.10)
else:
    new_conf_5=old_conf_5.copy()
    new_conf_10=old_conf_10.copy()
    p_candidate_test=None
    print('Development rejected Dandhi-only transfer; skipped all benchmark RGB inference')

policies={
    'baseline_conf_10':old_conf_10,
    'candidate_conf_10':new_conf_10,
    'baseline_dual_5_5':old_conf_5|c11_5,
    'candidate_dual_5_5':new_conf_5|c11_5,
    'C11_10':c11_10,
}
must(all(len(x)==12510 for x in policies.values()),'Policy length mismatch')
must(np.array_equal(policies['C11_10'],saved_flag('refer_C11_10')),
     'Frozen C11 predictions were unexpectedly modified')


## 6. Complete joint grade-or-refer evaluation (including paired newly triggered synthetic alerts)


In [ ]:
res=TEST[['sample_id','source_id','source_group','kind','label','family','severity','source_size','source_variety']].copy()
res['baseline_grade']=existing.predicted_size.to_numpy()
res['candidate_grade']=(np.asarray(LABELS)[p_candidate_test.argmax(1)]
                        if alpha>0 else res.baseline_grade.to_numpy().copy())
res['baseline_error']=res.baseline_grade.ne(res.source_size)
res['candidate_error']=res.candidate_grade.ne(res.source_size)
for name,flag in policies.items():res['refer_'+name]=flag
res.to_csv(OUT/'targeted_variant_predictions.csv',index=False)

clean=res.loc[TEST_CLEAN].set_index('source_id')
must(len(clean)==834 and clean.index.is_unique,'Clean source identity mismatch')
summary=[]
for name,flags in policies.items():
    use_candidate=name.startswith('candidate')
    error=res['candidate_error'].to_numpy(bool) if use_candidate else res['baseline_error'].to_numpy(bool)
    clean_flag_by_source=clean['refer_'+name]
    originally_referred=res.source_id.map(clean_flag_by_source).to_numpy(bool)
    edit_eligible=TEST_EDIT & ~originally_referred
    alert_trigger=edit_eligible & flags
    accepted=TEST_CLEAN & ~flags
    new_edit_error=TEST_EDIT & ~res.source_id.map(clean['candidate_error' if use_candidate else 'baseline_error']).to_numpy(bool) & error
    summary.append({
       'policy':name,
       'clean_accuracy':float(1-error[TEST_CLEAN].mean()),
       'actual_clean_referral':float(flags[TEST_CLEAN].mean()),
       'synthetic_recall':float(flags[TEST_EDIT].mean()),
       'incremental_synthetic_recall':float(alert_trigger.sum()/max(1,edit_eligible.sum())),
       'benign_referral':float(flags[TEST_BENIGN].mean()),
       'accepted_clean_error':float(error[accepted].mean()),
       'clean_errors_caught':int((flags & TEST_CLEAN & error).sum()),
       'new_edit_errors':int(new_edit_error.sum()),
       'new_edit_errors_referred':int((flags & new_edit_error).sum())
    })
metrics=pd.DataFrame(summary)
metrics.to_csv(OUT/'targeted_joint_referral_metrics.csv',index=False)
display(metrics.round(4))

# Bootstrap source-level contrasts. One original fruit and its 14 variants are one unit.
source_ids=np.sort(res.source_id.unique())
must(len(source_ids)==834 and res.groupby('source_id').size().eq(15).all(),
     'Broken source-pair structure')
rng=np.random.default_rng(20261023)
ids_to_i={source_id:i for i,source_id in enumerate(source_ids)}
index_of_source=res.source_id.map(ids_to_i).to_numpy(dtype=int)
bootstrap_rows=[]
for nm_old,nm_new in [('baseline_conf_10','candidate_conf_10'),
                      ('baseline_dual_5_5','candidate_dual_5_5')]:
    a=res['refer_'+nm_old].to_numpy(bool);b=res['refer_'+nm_new].to_numpy(bool)
    err_a=res.baseline_error.to_numpy(bool);err_b=res.candidate_error.to_numpy(bool)
    for metric in ('synthetic_recall','clean_referral','accepted_clean_error','incremental_synthetic_recall'):
        # Sampling *sources with replacement* requires source multiplicity, not just np.isin.
        base_num=np.zeros(834);new_num=np.zeros(834);base_den=np.zeros(834);new_den=np.zeros(834)
        for source_id,i in ids_to_i.items():
            mask=index_of_source==i
            for flag,err,num,den in [(a,err_a,base_num,base_den),(b,err_b,new_num,new_den)]:
                if metric=='synthetic_recall':m=mask & TEST_EDIT;n=flag[m].sum();d=m.sum()
                elif metric=='clean_referral':m=mask & TEST_CLEAN;n=flag[m].sum();d=m.sum()
                elif metric=='accepted_clean_error':m=mask & TEST_CLEAN & ~flag;n=err[m].sum();d=m.sum()
                else:
                    cf=bool(flag[mask & TEST_CLEAN][0]);m=mask & TEST_EDIT & (not cf)
                    n=flag[m].sum();d=m.sum()
                num[i]=n;den[i]=d
        observed=new_num.sum()/max(1,new_den.sum())-base_num.sum()/max(1,base_den.sum())
        samples=rng.integers(0,len(source_ids),(800,len(source_ids)))
        aa=base_num[samples].sum(1)/np.maximum(1,base_den[samples].sum(1))
        bb=new_num[samples].sum(1)/np.maximum(1,new_den[samples].sum(1))
        lo,hi=np.quantile(bb-aa,[.025,.975])
        bootstrap_rows.append({'comparison':f'{nm_new} minus {nm_old}','metric':metric,
              'delta_pp':100*observed,'ci_low_pp':100*lo,'ci_high_pp':100*hi})
bootstrap=pd.DataFrame(bootstrap_rows)
bootstrap.to_csv(OUT/'targeted_paired_bootstrap.csv',index=False)
display(bootstrap.round(3))

variety_rows=[]
for variety_name,g in clean.groupby('source_variety'):
    variety_rows.append({'variety':variety_name,'n':len(g),
      'baseline_accuracy':float(1-g.baseline_error.mean()),
      'candidate_accuracy':float(1-g.candidate_error.mean())})
variety=pd.DataFrame(variety_rows)
variety.to_csv(OUT/'targeted_variety_diagnostics.csv',index=False)
display(variety.round(4))

fig,axs=plt.subplots(1,2,figsize=(11,4))
for _,r in metrics.iterrows():
    axs[0].scatter(100*r.actual_clean_referral,100*r.synthetic_recall,label=r.policy)
    axs[1].scatter(100*r.actual_clean_referral,100*r.accepted_clean_error,label=r.policy)
axs[0].set(ylabel='Synthetic referral (%)',xlabel='Original referral (%)',title='Higher synthetic recall is better')
axs[1].set(ylabel='Accepted grading error (%)',xlabel='Original referral (%)',title='Lower grading error is better')
for ax in axs:ax.grid(alpha=.3)
axs[1].legend(fontsize=7)
plt.tight_layout();plt.savefig(OUT/'targeted_grade_or_refer.png',dpi=140);plt.show()

report={'revision':REVISION,'status':'EXPLORATORY_COMPLETED',
 'development_decision':'TARGETED_ACCEPTED' if alpha>0 else 'TARGETED_REJECTED',
 'blend_weight':alpha,'checkpoint_reused':str(model_file),'checkpoint_sha256':candidate_ckpt_sha,
 'pretrained_sha256':weight_hash,'development_selection':selection,
 'target_variety':'Dandhi','non_target_varieties_unchanged':True,
 'test_used_for_training_or_selection':False,'manual_annotation':False,'manual_data_preparation':False,
 'defect_labels_in_training':False,'frozen_C11_modified':False,'gpu_training':False,'any_training':False,
 'limitations':['Only synthetic defects for quantitative detection, no verified real-defect test',
  'Existing V4 locked test repeatedly examined, so comparisons remain exploratory',
  'Dataset sizes are categorical image labels, not verified physical mm measurements'],
 'artifacts':['prior_saved_rgb_checkpoint_reused',
 'development_only_selection.json','targeted_variant_predictions.csv',
 'targeted_joint_referral_metrics.csv','targeted_paired_bootstrap.csv',
 'targeted_variety_diagnostics.csv','targeted_grade_or_refer.png']}
json_write(OUT/'RESULTS_COMPLETE.json',report)
print('EXPERIMENT COMPLETED:',OUT)
print('Development-only decision:',report['development_decision'],'candidate blend:',alpha)
print('Candidate analysis completed using the specified frozen reference models.')
